In [2]:
%pylab inline
import os
import gc
import matplotlib
import numpy as np
import astropy.io.fits as pyfits
import astropy.table as astTab
import sim_analysis_utilities as utilities

Populating the interactive namespace from numpy and matplotlib


In [3]:
cd wrkDir

/work/xiangchong.li/work/galSim-HSC


In [4]:
def chunkNList(seq, num):
    avg = len(seq) / float(num)
    out = []
    last = 0.0
    while last < len(seq):
        out.append(seq[int(last):int(last + avg)])
        last += avg
    return out

def delcolnull(fits):
    colns   =   fits.colnames
    colns2  =   [cn for cn in colns if "_isnull" not in cn]
    fits    =   fits[colns2]
    return fits

In [5]:
info=astTab.Table.read('fieldTractInfoS19.csv')
info2=chunkNList(info,20)

In [ ]:
for fieldname in utilities.field_names:
    if fieldname=='XMM':
        continue
    cat=astTab.Table.read('./s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_others/%s_pz.fits'%fieldname)
    catAll=[]
    for ii,ifo in enumerate(info2):
        if fieldname in set(ifo['field']):
            sub=astTab.Table.read('./database/s19-photoz/sql/%d.fits' %ii)
            sub=delcolnull(sub)
            catAll.append(astTab.join(left=sub,right=cat,keys='object_id',join_type='inner'))
            del sub
            gc.collect()
    catOut=astTab.vstack(catAll)
    catOut.sort('object_id')
    assert np.all(catOut['object_id']==cat['object_id']),'failed'
    catOut.write('./s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_others/%s_pz_2.fits'%fieldname)
    del cat
    gc.collect()